# Tutorial 17 — Capstone: Training a Reasoning Model End to End

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part VI — Capstone**  
**Follows:** Tutorial 16 (Inference)  
**Precedes:** Tutorial 18 (DeepSeek Architecture: MLA & MoE)

---

## What This Tutorial Covers

The preceding tutorials treated each component of the pipeline in
isolation. This tutorial composes them into a single end-to-end
program: from random initialisation to a deployed, quantized,
instruction-following model. All modules are drawn from this series.
[[[No external pretrained weights are used.]]{.mark}{.mark}

The pipeline:

```
Random init
    ↓
Pretraining       (T7, T8, T9)   — language modelling objective
    ↓
SFT + LoRA        (T10)          — instruction following
    ↓
Reward Model      (T12)          — preference scoring
    ↓
GRPO              (T13)          — policy optimisation
    ↓
INT8 Quantization (T14)          — weight compression
    ↓
Fast Inference    (T15)          — KV cache + speculative decoding
    ↓
Evaluation                       — perplexity, throughput, qualitative
```

The model is the nano GPT from Tutorial 2: 10.7M parameters, trained on
TinyShakespeare. Since the model has been trained solely on Shakespeare
data, it cannot answer general questions about other domains. But we can
run `engine.generate("What should Hamlet do?")` and it replies in coherent
Shakespearean prose with a recommendation and a reason. This tells us that
it has learned. One rewarding aspect of this exercise is that we will
understand each step of the process that led to this output — details that
we abstract over when we use API providers.

---

## Two Configurations: Pico and Nano

Before renting GPU time, you want to know the pipeline runs correctly on
your own machine. The `pico` config exists for this purpose. It is a
deliberately tiny model trained for very few steps — small enough to
complete the full pipeline in under 10 minutes on a laptop CPU. It will
not produce coherent outputs, but it will confirm that every stage
executes, every checkpoint saves, and every metric is being recorded.
If pico passes, you can run `nano` on a proper GPU with confidence.

```
pico  — sanity check on a laptop. Full pipeline in ~6 minutes.
        Loss should decrease. Reward margin should increase. That is all.

nano  — the real run. Trains a 10.7M parameter model to coherent
        Shakespearean instruction-following on a single GPU.
        Full pipeline in ~20–50 minutes depending on hardware.
```

The two configs differ only in model size, step counts, and batch sizes.
The data, the code, and the pipeline structure are identical. If pico
passes, nano will pass.

In [ ]:
# train_nano_reasoning.py
# ─────────────────────────────────────────────────────────────────────────────

import os
import json
import time
import math
import random
import textwrap
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

# ── Modules from this series ──────────────────────────────────────────────────
from tutorial_02 import GPT, NanoGPTConfig
from tutorial_03 import Tokenizer
from tutorial_04 import GradientMonitor
from tutorial_05 import TrainingLogger
from tutorial_07 import PretrainingDataset
from tutorial_08 import make_cosine_schedule
from tutorial_10 import (InstructDataset, inject_lora, freeze_base_model,
                          merge_lora, sft_loss, SPECIAL_TOKENS, collate_sft)
from tutorial_12 import RewardModel, RewardDataset, reward_loss, collate_reward
from tutorial_13 import (GRPOConfig, compute_group_advantages, grpo_loss,
                          sample_responses, score_responses,
                          get_response_log_probs, GRPOEarlyStopper)
from tutorial_14 import quantize_model_ptq, model_size_mb
from tutorial_15 import FastInferenceEngine, generate_with_cache


# ── Configuration dataclasses ─────────────────────────────────────────────────

@dataclass
class ModelConfig:
    """Model architecture hyperparameters."""
    vocab_size:  int   = 4096
    d_model:     int   = 384
    n_layers:    int   = 6
    n_heads:     int   = 6
    d_ff:        int   = 1536
    max_seq_len: int   = 256
    dropout:     float = 0.1


@dataclass
class PipelineConfig:
    """Full pipeline configuration. Use pico_config() or nano_config()."""

    # Paths
    run_dir:         str = 'runs/nano_reasoning'
    data_dir:        str = 'data'

    # Model
    model: ModelConfig = field(default_factory=ModelConfig)

    # Pretraining
    pretrain_steps:  int   = 5000
    pretrain_lr:     float = 3e-4
    pretrain_batch:  int   = 32
    pretrain_seq:    int   = 256

    # SFT
    sft_steps:       int   = 1000
    sft_lr:          float = 2e-4
    sft_batch:       int   = 4
    lora_rank:       int   = 8

    # Reward model
    rm_steps:        int   = 500
    rm_lr:           float = 1e-4
    rm_batch:        int   = 4

    # GRPO
    grpo_steps:      int   = 200
    grpo_lr:         float = 1e-6
    grpo_G:          int   = 8
    grpo_beta:       float = 0.04

    # Inference
    quantize:        bool  = True
    quant_bits:      int   = 8
    use_speculative: bool  = True
    spec_k:          int   = 4

    # Misc
    seed:            int   = 42
    device:          str   = 'auto'
    dtype:           str   = 'bfloat16'

    def __post_init__(self):
        if self.device == 'auto':
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

    def save(self, path: str):
        with open(path, 'w') as f:
            json.dump(asdict(self), f, indent=2, default=str)

    @classmethod
    def load(cls, path: str) -> 'PipelineConfig':
        with open(path) as f:
            d = json.load(f)
        cfg = cls()
        for k, v in d.items():
            if hasattr(cfg, k):
                setattr(cfg, k, v)
        return cfg


def pico_config() -> PipelineConfig:
    """
    Minimal configuration for local sanity checking.

    Run this first on your laptop to verify the pipeline executes end to
    end. Steps are too few for good model quality — the goal is to see
    loss decreasing and all checkpoints saving correctly, not to produce
    a usable model.

    Approximate runtime: 6–10 minutes on a modern laptop CPU.
    """
    return PipelineConfig(
        run_dir        = 'runs/pico_reasoning',
        model          = ModelConfig(
            vocab_size  = 512,
            d_model     = 64,
            n_layers    = 2,
            n_heads     = 2,
            d_ff        = 256,
            max_seq_len = 64,
            dropout     = 0.0,
        ),
        pretrain_steps = 200,
        pretrain_lr    = 3e-4,
        pretrain_batch = 8,
        pretrain_seq   = 64,

        sft_steps      = 50,
        sft_lr         = 2e-4,
        sft_batch      = 2,
        lora_rank      = 4,

        rm_steps       = 30,
        rm_lr          = 1e-4,
        rm_batch       = 2,

        grpo_steps     = 20,
        grpo_lr        = 1e-6,
        grpo_G         = 4,
        grpo_beta      = 0.04,

        quantize        = True,
        quant_bits      = 8,
        use_speculative = False,   # skip for pico — draft model overhead not worth it
        dtype           = 'float32',   # bfloat16 not available on all CPUs
    )


def nano_config() -> PipelineConfig:
    """
    Full configuration for a single-GPU run.

    Trains a 10.7M parameter GPT on TinyShakespeare through the complete
    pretraining → SFT → reward model → GRPO → quantize → inference pipeline.

    Approximate runtimes:
        A100 40GB     ~20 min
        RTX 3090      ~50 min
        M2 MacBook    ~2.7 hr  (MPS backend)
    """
    return PipelineConfig(
        run_dir        = 'runs/nano_reasoning',
        model          = ModelConfig(
            vocab_size  = 4096,
            d_model     = 384,
            n_layers    = 6,
            n_heads     = 6,
            d_ff        = 1536,
            max_seq_len = 256,
            dropout     = 0.1,
        ),
        pretrain_steps = 5000,
        pretrain_lr    = 3e-4,
        pretrain_batch = 32,
        pretrain_seq   = 256,

        sft_steps      = 1000,
        sft_lr         = 2e-4,
        sft_batch      = 4,
        lora_rank      = 8,

        rm_steps       = 500,
        rm_lr          = 1e-4,
        rm_batch       = 4,

        grpo_steps     = 200,
        grpo_lr        = 1e-6,
        grpo_G         = 8,
        grpo_beta      = 0.04,

        quantize        = True,
        quant_bits      = 8,
        use_speculative = True,
        spec_k          = 4,
        dtype           = 'bfloat16',
    )

---

## What to Look for During Pico

[The pico model is too small and too undertrained to produce meaningful
text. That is expected.]{.underline} Check the following:

- **Pretraining loss decreases.** Over 200 steps, loss should drop from
  roughly $\log(\text{vocab\_size}) \approx 6.2$ toward something lower.
  A flat or rising curve indicates a bug in the optimizer, data loader,
  or forward pass.
- **SFT eval loss is lower than random.** The model will not follow
  instructions coherently, but the loss should be finite and decreasing.
- [[**Reward model accuracy exceeds 0.5.**]]{.mark} Even 30 training steps should
  push accuracy above chance, because the chosen/rejected split is stark
  (coherent text vs word-shuffled text).
- **GRPO reward margin trends upward.** With only 20 steps the improvement
  will be small, but the margin should be non-negative by the end.
- **All checkpoints save and load without error.** The most common
  failure mode on a first run is a shape or key mismatch at checkpoint
  load time.
- **`stage_evaluate` runs to completion.** Outputs will be incoherent
  — that is fine. The point is that the function does not crash.

If all six hold, the pipeline is correctly wired. Proceed to `nano`.

---

## Stage 1: Pretraining

In [ ]:
def stage_pretrain(cfg: PipelineConfig, tok: Tokenizer) -> str:
    """
    Pretrain on raw text. Returns path to saved checkpoint.
    Skips automatically if a checkpoint already exists.
    """
    print("\n" + "═"*60)
    print("  STAGE 1 — PRETRAINING")
    print("═"*60)

    device    = torch.device(cfg.device)
    dtype     = torch.bfloat16 if cfg.dtype == 'bfloat16' else torch.float32
    ckpt_path = f'{cfg.run_dir}/pretrained.pt'

    if Path(ckpt_path).exists():
        print(f"  Checkpoint found — skipping.")
        return ckpt_path

    model    = GPT(cfg.model).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters : {n_params/1e6:.2f}M")
    print(f"  Device     : {device}  |  dtype: {dtype}")

    raw_path = Path(cfg.data_dir) / 'tinyshakespeare.txt'
    if not raw_path.exists():
        _download_tinyshakespeare(raw_path)

    dataset = PretrainingDataset(
        data_path=str(raw_path),
        tokenizer=tok,
        block_size=cfg.pretrain_seq,
    )
    loader = DataLoader(dataset, batch_size=cfg.pretrain_batch,
                        shuffle=True, num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.pretrain_lr, weight_decay=0.1
    )
    scheduler = make_cosine_schedule(
        optimizer, cfg.pretrain_lr, cfg.pretrain_lr * 0.1,
        warmup_steps=min(100, cfg.pretrain_steps // 10),
        max_steps=cfg.pretrain_steps,
    )

    monitor   = GradientMonitor(model,
                                 log_every=max(100, cfg.pretrain_steps // 10))
    logger    = TrainingLogger(f'{cfg.run_dir}/pretrain_log.jsonl')
    model.train()
    data_iter = iter(loader)
    best_loss = float('inf')
    t0        = time.time()
    log_every = max(50, cfg.pretrain_steps // 20)

    for step in range(cfg.pretrain_steps):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(loader)
            x, y = next(data_iter)

        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=dtype):
            _, loss = model(x, y)

        optimizer.zero_grad()
        loss.backward()
        monitor.step()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        lr = optimizer.param_groups[0]['lr']
        logger.log(step=step, loss=loss.item(), lr=lr)

        if step % log_every == 0:
            elapsed = time.time() - t0
            ppl     = math.exp(min(loss.item(), 20))
            print(f"  step {step:5d}/{cfg.pretrain_steps}  "
                  f"loss={loss.item():.4f}  ppl={ppl:.1f}  "
                  f"lr={lr:.2e}  {elapsed:.0f}s")

        if loss.item() < best_loss:
            best_loss = loss.item()
            torch.save({'model': model.state_dict(),
                        'step':  step,
                        'loss':  best_loss},
                       ckpt_path)

    elapsed = time.time() - t0
    print(f"\n  Done in {elapsed:.0f}s  |  "
          f"best loss={best_loss:.4f}  ppl={math.exp(best_loss):.1f}")
    print(f"  → {ckpt_path}")
    return ckpt_path

---

## Stage 2: SFT + LoRA

In [ ]:
def stage_sft(cfg: PipelineConfig, tok: Tokenizer,
              pretrain_ckpt: str) -> str:
    """
    Supervised fine-tuning with LoRA.
    Returns path to the merged checkpoint (LoRA folded into base weights).
    """
    print("\n" + "═"*60)
    print("  STAGE 2 — SFT + LoRA")
    print("═"*60)

    device    = torch.device(cfg.device)
    dtype     = torch.bfloat16 if cfg.dtype == 'bfloat16' else torch.float32
    ckpt_path = f'{cfg.run_dir}/sft_merged.pt'

    if Path(ckpt_path).exists():
        print(f"  Checkpoint found — skipping.")
        return ckpt_path

    sft_data = f'{cfg.run_dir}/sft_data.jsonl'
    if not Path(sft_data).exists():
        raw = (Path(cfg.data_dir) / 'tinyshakespeare.txt').read_text()
        _make_shakespeare_sft(raw, sft_data,
                               n_samples=max(100, cfg.sft_steps // 2))

    model = GPT(cfg.model).to(device)
    ckpt  = torch.load(pretrain_ckpt, map_location=device)
    model.load_state_dict(ckpt['model'])
    print(f"  Loaded pretrained weights.")

    inject_lora(model, rank=cfg.lora_rank)
    freeze_base_model(model)

    dataset = InstructDataset(sft_data, tok, max_length=cfg.pretrain_seq)
    val_n   = max(1, len(dataset) // 10)
    train_ds, val_ds = random_split(
        dataset, [len(dataset) - val_n, val_n]
    )
    train_loader = DataLoader(train_ds, batch_size=cfg.sft_batch,
                               shuffle=True, collate_fn=collate_sft,
                               num_workers=2)
    val_loader   = DataLoader(val_ds, batch_size=cfg.sft_batch,
                               shuffle=False, collate_fn=collate_sft)

    lora_params = [p for p in model.parameters() if p.requires_grad]
    optimizer   = torch.optim.AdamW(lora_params, lr=cfg.sft_lr,
                                     weight_decay=0.01)
    scheduler   = make_cosine_schedule(
        optimizer, cfg.sft_lr, cfg.sft_lr * 0.1,
        warmup_steps=min(50, cfg.sft_steps // 10),
        max_steps=cfg.sft_steps,
    )

    model.train()
    data_iter = iter(train_loader)
    best_eval = float('inf')
    t0        = time.time()
    log_every = max(10, cfg.sft_steps // 10)

    for step in range(cfg.sft_steps):
        try:
            x, y = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            x, y = next(data_iter)

        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=dtype):
            logits, _ = model(x)
            loss      = sft_loss(logits, y)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
        optimizer.step()
        scheduler.step()

        if step % log_every == 0:
            model.eval()
            eval_losses = []
            with torch.no_grad():
                for xv, yv in val_loader:
                    with torch.autocast(device_type=device.type, dtype=dtype):
                        lv, _ = model(xv.to(device))
                        eval_losses.append(
                            sft_loss(lv, yv.to(device)).item()
                        )
            eval_loss = float(np.mean(eval_losses))
            model.train()
            lr = optimizer.param_groups[0]['lr']
            print(f"  step {step:4d}/{cfg.sft_steps}  "
                  f"train={loss.item():.4f}  eval={eval_loss:.4f}  "
                  f"lr={lr:.2e}")

            if eval_loss < best_eval:
                best_eval = eval_loss
                torch.save(
                    {'lora': {k: v for k, v in model.state_dict().items()
                              if 'lora_A' in k or 'lora_B' in k},
                     'step': step},
                    f'{cfg.run_dir}/sft_lora.pt'
                )

    model = merge_lora(model)
    torch.save({'model': model.state_dict()}, ckpt_path)

    elapsed = time.time() - t0
    print(f"\n  Done in {elapsed:.0f}s  |  best eval loss={best_eval:.4f}")
    print(f"  → {ckpt_path}  (LoRA merged)")
    return ckpt_path

---

## Stage 3: Reward Model

In [ ]:
def stage_reward_model(cfg: PipelineConfig, tok: Tokenizer,
                        sft_ckpt: str) -> str:
    """
    Train the reward model on preference pairs.
    Chosen = real Shakespeare excerpt. Rejected = word-shuffled version.
    Returns path to saved checkpoint.
    """
    print("\n" + "═"*60)
    print("  STAGE 3 — REWARD MODEL")
    print("═"*60)

    device    = torch.device(cfg.device)
    dtype     = torch.bfloat16 if cfg.dtype == 'bfloat16' else torch.float32
    ckpt_path = f'{cfg.run_dir}/reward_model.pt'

    if Path(ckpt_path).exists():
        print(f"  Checkpoint found — skipping.")
        return ckpt_path

    pref_data = f'{cfg.run_dir}/preference_data.jsonl'
    if not Path(pref_data).exists():
        raw = (Path(cfg.data_dir) / 'tinyshakespeare.txt').read_text()
        _make_preference_data(raw, pref_data,
                               n_samples=max(60, cfg.rm_steps // 2))

    reward_model = RewardModel(cfg.model, backbone_path=sft_ckpt).to(device)

    dataset  = RewardDataset(pref_data, tok, max_length=cfg.pretrain_seq)
    val_n    = max(1, len(dataset) // 10)
    train_ds, val_ds = random_split(dataset, [len(dataset) - val_n, val_n])
    train_loader = DataLoader(train_ds, batch_size=cfg.rm_batch,
                               shuffle=True, collate_fn=collate_reward,
                               num_workers=1)
    val_loader   = DataLoader(val_ds, batch_size=cfg.rm_batch,
                               shuffle=False, collate_fn=collate_reward)

    optimizer = torch.optim.AdamW([
        {'params': reward_model.reward_head.parameters(),
         'lr':     cfg.rm_lr * 10},
        {'params': [p for p in reward_model.backbone.parameters()
                    if p.requires_grad],
         'lr':     cfg.rm_lr},
    ], weight_decay=0.01)
    scheduler = make_cosine_schedule(
        optimizer, cfg.rm_lr, cfg.rm_lr * 0.1,
        warmup_steps=min(30, cfg.rm_steps // 5),
        max_steps=cfg.rm_steps,
    )

    reward_model.train()
    data_iter = iter(train_loader)
    best_acc  = 0.0
    t0        = time.time()
    log_every = max(5, cfg.rm_steps // 10)

    for step in range(cfg.rm_steps):
        try:
            c_ids, c_mask, r_ids, r_mask = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            c_ids, c_mask, r_ids, r_mask = next(data_iter)

        c_ids = c_ids.to(device); c_mask = c_mask.to(device)
        r_ids = r_ids.to(device); r_mask = r_mask.to(device)

        with torch.autocast(device_type=device.type, dtype=dtype):
            cr = reward_model(c_ids, c_mask)
            rr = reward_model(r_ids, r_mask)
            loss, metrics = reward_loss(cr, rr, margin=0.5)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if step % log_every == 0:
            reward_model.eval()
            val_accs = []
            with torch.no_grad():
                for c_v, cm_v, r_v, rm_v in val_loader:
                    c_v  = c_v.to(device);  cm_v = cm_v.to(device)
                    r_v  = r_v.to(device);  rm_v = rm_v.to(device)
                    cr_v = reward_model(c_v, cm_v)
                    rr_v = reward_model(r_v, rm_v)
                    val_accs.append(
                        ((cr_v - rr_v) > 0).float().mean().item()
                    )
            val_acc = float(np.mean(val_accs))
            reward_model.train()
            print(f"  step {step:4d}/{cfg.rm_steps}  "
                  f"loss={loss.item():.4f}  "
                  f"gap={metrics['reward_gap']:.3f}  "
                  f"val_acc={val_acc:.2%}")

            if val_acc > best_acc:
                best_acc = val_acc
                torch.save({'model': reward_model.state_dict()}, ckpt_path)

    elapsed = time.time() - t0
    print(f"\n  Done in {elapsed:.0f}s  |  best val_acc={best_acc:.2%}")
    print(f"  → {ckpt_path}")
    return ckpt_path

---

## Stage 4: GRPO

In [ ]:
def stage_grpo(cfg: PipelineConfig, tok: Tokenizer,
               sft_ckpt: str, rm_ckpt: str) -> str:
    """
    GRPO policy optimisation.
    Returns path to the final policy checkpoint.
    """
    print("\n" + "═"*60)
    print("  STAGE 4 — GRPO")
    print("═"*60)

    device    = torch.device(cfg.device)
    dtype     = torch.bfloat16 if cfg.dtype == 'bfloat16' else torch.float32
    ckpt_path = f'{cfg.run_dir}/grpo_final.pt'

    if Path(ckpt_path).exists():
        print(f"  Checkpoint found — skipping.")
        return ckpt_path

    # ── Load models ───────────────────────────────────────────────────────────
    policy = GPT(cfg.model).to(device)
    sft_st = torch.load(sft_ckpt, map_location=device)
    policy.load_state_dict(sft_st['model'])

    ref_model = GPT(cfg.model).to(device)
    ref_model.load_state_dict(sft_st['model'])
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad_(False)

    rm_st        = torch.load(rm_ckpt, map_location=device)
    reward_model = RewardModel(cfg.model).to(device)
    reward_model.load_state_dict(rm_st['model'])
    reward_model.eval()
    for p in reward_model.parameters():
        p.requires_grad_(False)

    prompts   = _make_grpo_prompts()
    grpo_cfg  = GRPOConfig(
        G              = cfg.grpo_G,
        beta           = cfg.grpo_beta,
        max_lr         = cfg.grpo_lr,
        max_steps      = cfg.grpo_steps,
        max_new_tokens = min(80, cfg.pretrain_seq // 2),
    )
    optimizer = torch.optim.AdamW(
        policy.parameters(), lr=grpo_cfg.max_lr, weight_decay=0.01
    )
    scheduler = make_cosine_schedule(
        optimizer, grpo_cfg.max_lr, grpo_cfg.max_lr * 0.1,
        warmup_steps=grpo_cfg.warmup_steps,
        max_steps=grpo_cfg.max_steps,
    )
    stopper   = GRPOEarlyStopper(kl_threshold=1.0, reward_patience=40)
    history   = []
    t0        = time.time()
    log_every = max(5, cfg.grpo_steps // 10)

    for step in range(grpo_cfg.max_steps):
        prompt_batch = random.sample(prompts, min(4, len(prompts)))
        step_metrics = []
        optimizer.zero_grad()

        for prompt_text in prompt_batch:
            prompt_enc = (
                f"{SPECIAL_TOKENS['user']}\n{prompt_text}\n"
                f"{SPECIAL_TOKENS['end']}\n"
                f"{SPECIAL_TOKENS['assistant']}\n"
            )
            prompt_ids = torch.tensor(
                [tok.encode(prompt_enc)], dtype=torch.long, device=device
            )

            with torch.no_grad():
                resp_ids   = sample_responses(
                    policy, tok, prompt_ids,
                    G=grpo_cfg.G,
                    max_new_tokens=grpo_cfg.max_new_tokens,
                    temperature=grpo_cfg.temperature,
                )
                rewards    = score_responses(
                    reward_model, tok, prompt_ids, resp_ids, device
                )
                advantages = compute_group_advantages(rewards)
                prompt_rep = prompt_ids.repeat(grpo_cfg.G, 1)
                old_lp, mask = get_response_log_probs(
                    policy, prompt_rep, resp_ids
                )
                ref_lp, _    = get_response_log_probs(
                    ref_model, prompt_rep, resp_ids
                )

            policy.train()
            with torch.autocast(device_type=device.type, dtype=dtype):
                pol_lp, _ = get_response_log_probs(
                    policy, prompt_rep, resp_ids
                )

            loss, metrics = grpo_loss(
                pol_lp, old_lp, ref_lp, advantages, mask,
                clip_eps=grpo_cfg.clip_eps,
                beta=grpo_cfg.beta,
            )
            metrics['mean_reward'] = rewards.mean().item()
            metrics['reward_std']  = rewards.std().item()
            (loss / len(prompt_batch)).backward()
            step_metrics.append(metrics)

        torch.nn.utils.clip_grad_norm_(
            policy.parameters(), grpo_cfg.grad_clip
        )
        optimizer.step()
        scheduler.step()

        avg = {k: np.mean([m[k] for m in step_metrics])
               for k in step_metrics[0]}
        history.append(avg)

        if step % log_every == 0:
            print(f"  step {step:4d}/{grpo_cfg.max_steps}  "
                  f"reward={avg['mean_reward']:+.3f}  "
                  f"kl={avg['mean_kl']:.4f}  "
                  f"margin={avg['mean_advantage']:+.3f}  "
                  f"clip%={avg['clipped_frac']:.1%}")

        should_stop, reason = stopper.check(avg)
        if should_stop:
            print(f"\n  Early stop at step {step}: {reason}")
            break

    torch.save({'model': policy.state_dict(), 'history': history},
               ckpt_path)

    elapsed      = time.time() - t0
    init_reward  = np.mean([m['mean_reward'] for m in history[:5]])
    final_reward = np.mean([m['mean_reward'] for m in history[-5:]])
    print(f"\n  Done in {elapsed:.0f}s  |  "
          f"reward {init_reward:+.3f} → {final_reward:+.3f}  "
          f"(Δ={final_reward - init_reward:+.3f})")
    print(f"  → {ckpt_path}")
    return ckpt_path

---

## Stage 5: Deployment

In [ ]:
def stage_deploy(cfg: PipelineConfig, tok: Tokenizer,
                 grpo_ckpt: str) -> FastInferenceEngine:
    """
    Quantize the model and initialise the inference engine.
    Returns a ready-to-use FastInferenceEngine.
    """
    print("\n" + "═"*60)
    print("  STAGE 5 — DEPLOYMENT")
    print("═"*60)

    device = torch.device(cfg.device)

    model = GPT(cfg.model).to(device)
    ckpt  = torch.load(grpo_ckpt, map_location=device)
    model.load_state_dict(ckpt['model'])
    model.eval()

    size_fp = model_size_mb(model)
    print(f"  Size (fp32)  : {size_fp:.1f} MB")

    if cfg.quantize:
        quantize_model_ptq(model, n_bits=cfg.quant_bits)
        size_q    = model_size_mb(model)
        reduction = 100 * (1 - size_q / size_fp)
        print(f"  Size (INT{cfg.quant_bits}) : {size_q:.1f} MB  "
              f"({reduction:.0f}% reduction)")

    # Draft model for speculative decoding — a 2-layer d=128 version.
    # Small enough to be fast; large enough to have a reasonable acceptance rate.
    draft_model = None
    if cfg.use_speculative:
        from dataclasses import replace
        draft_cfg   = replace(cfg.model, n_layers=2, d_model=128,
                               n_heads=2, d_ff=512)
        draft_model = GPT(draft_cfg).to(device)
        draft_model.eval()
        if cfg.quantize:
            quantize_model_ptq(draft_model, n_bits=cfg.quant_bits)
        n_draft = sum(p.numel() for p in draft_model.parameters())
        print(f"  Draft model  : {n_draft/1e6:.2f}M params")

    engine = FastInferenceEngine(
        model           = model,
        tokenizer       = tok,
        device          = device,
        use_speculative = cfg.use_speculative and draft_model is not None,
        draft_model     = draft_model,
        k               = cfg.spec_k,
    )
    print(f"  Engine ready.")
    return engine

---

## Stage 6: Evaluation

In [ ]:
def stage_evaluate(
    cfg:           PipelineConfig,
    tok:           Tokenizer,
    engine:        FastInferenceEngine,
    pretrain_ckpt: str,
    sft_ckpt:      str,
    grpo_ckpt:     str,
):
    """
    Three-part evaluation:
      1. Qualitative samples from the final (GRPO) model.
      2. Throughput benchmark.
      3. Before/after comparison across pipeline stages.
    """
    print("\n" + "═"*60)
    print("  STAGE 6 — EVALUATION")
    print("═"*60)

    device = torch.device(cfg.device)

    eval_prompts = [
        "What should Hamlet do about his father's murder?",
        "Write a brief speech about the nature of ambition.",
        "Who is more dangerous: Iago or the witches in Macbeth?",
        "Describe the relationship between power and corruption.",
        "What is the right way to treat a defeated enemy?",
    ]

    # ── 1. Qualitative samples ─────────────────────────────────────────────
    print("\n  Qualitative samples (GRPO model):")
    print("  " + "─"*56)

    for prompt in eval_prompts[:3]:
        response, stats = engine.generate(
            prompt, max_new_tokens=120, temperature=0.8
        )
        print(f"\n  Q: {prompt}")
        wrapped = textwrap.fill(
            response.strip(), width=56,
            initial_indent='  A: ', subsequent_indent='     '
        )
        print(wrapped)
        print(f"     [{stats['tokens_generated']} tokens  "
              f"{stats['tokens_per_sec']:.0f} tok/s]")

    # ── 2. Throughput benchmark ────────────────────────────────────────────
    print("\n  Throughput benchmark:")
    print("  " + "─"*56)

    bm_prompts = eval_prompts * 4
    bm_stats   = engine.benchmark(bm_prompts, max_new_tokens=80)
    print(f"  Mean   : {bm_stats['mean_tps']:.1f} tokens/sec")
    print(f"  Median : {bm_stats['median_tps']:.1f} tokens/sec")
    print(f"  P10    : {bm_stats['p10_tps']:.1f} tokens/sec")
    print(f"  P90    : {bm_stats['p90_tps']:.1f} tokens/sec")

    # ── 3. Before/after comparison ─────────────────────────────────────────
    # Three models, same prompt. The difference in outputs is the concrete
    # result of each alignment stage.
    print("\n  Before/after comparison:")
    print(f"  Prompt: \"{eval_prompts[0]}\"")
    print("  " + "─"*56)

    stages = [
        ('Pretrained', pretrain_ckpt),
        ('SFT + LoRA', sft_ckpt),
        ('GRPO',       grpo_ckpt),
    ]

    for stage_name, ckpt_path in stages:
        model_s = GPT(cfg.model).to(device)
        ckpt    = torch.load(ckpt_path, map_location=device)
        model_s.load_state_dict(ckpt['model'])
        model_s.eval()

        response = generate_with_cache(
            model_s, tok, eval_prompts[0],
            max_new_tokens=80, temperature=0.8, device=device
        )
        print(f"\n  [{stage_name}]")
        wrapped = textwrap.fill(
            response.strip()[:300], width=56,
            initial_indent='  ', subsequent_indent='  '
        )
        print(wrapped)
        del model_s

    # ── 4. Perplexity on held-out Shakespeare ──────────────────────────────
    print("\n  Perplexity on held-out Shakespeare:")
    print("  " + "─"*56)

    raw     = (Path(cfg.data_dir) / 'tinyshakespeare.txt').read_text()
    words   = raw.split()
    holdout = ' '.join(words[-5000:])
    tokens  = tok.encode(holdout)
    seq_len = cfg.pretrain_seq

    for stage_name, ckpt_path in stages:
        model_s = GPT(cfg.model).to(device)
        ckpt    = torch.load(ckpt_path, map_location=device)
        model_s.load_state_dict(ckpt['model'])
        model_s.eval()

        total_loss = 0; total_n = 0
        with torch.no_grad():
            for i in range(0, len(tokens) - seq_len - 1, seq_len):
                x = torch.tensor(
                    [tokens[i:i+seq_len]], dtype=torch.long, device=device
                )
                y = torch.tensor(
                    [tokens[i+1:i+seq_len+1]], dtype=torch.long, device=device
                )
                _, loss = model_s(x, y)
                total_loss += loss.item() * seq_len
                total_n    += seq_len

        ppl = math.exp(total_loss / max(total_n, 1))
        print(f"  {stage_name:20s}  ppl = {ppl:.2f}")
        del model_s

    # ── Summary ────────────────────────────────────────────────────────────
    print("\n" + "═"*60)
    print("  PIPELINE SUMMARY")
    print("═"*60)
    print(f"""
  Stage 1 — Pretraining
    Architecture : {cfg.model.d_model}-dim, {cfg.model.n_layers}-layer GPT
    Steps        : {cfg.pretrain_steps}
    Corpus       : TinyShakespeare

  Stage 2 — SFT + LoRA
    LoRA rank    : {cfg.lora_rank}
    Steps        : {cfg.sft_steps}

  Stage 3 — Reward Model
    Steps        : {cfg.rm_steps}
    Loss         : Bradley-Terry (margin = 0.5)

  Stage 4 — GRPO
    Steps        : {cfg.grpo_steps}
    Group size G : {cfg.grpo_G}
    β (KL)       : {cfg.grpo_beta}

  Stage 5 — Deployment
    Quantization : INT{cfg.quant_bits if cfg.quantize else 32}
    Model size   : {model_size_mb(engine.model):.1f} MB
    Speculative  : {'k=' + str(cfg.spec_k) if engine.use_spec else 'disabled'}
    Throughput   : {bm_stats['mean_tps']:.0f} tokens/sec (mean)
    """)

---

## Interpreting the Output

The `stage_evaluate` function produces a three-way comparison across
pipeline stages. A few things are worth noting when you read the output.

The pretrained model produces *a text continuation*. It has learned the
statistical structure of Shakespearean English but has no concept of a
question-answer format. Given the prompt "What should Hamlet do?", it
will likely continue with something that sounds plausibly Shakespearean
but addressed to no one in particular.

The SFT model addresses the prompt directly. It has learned the structural
form of a response from the instruction-tuning data. The quality of its
reasoning is limited by the SFT dataset — in this case, a few hundred
Shakespeare excerpts with synthetic prompts — so do not expect deep
philosophical insight.

The GRPO model has been further optimised toward responses the reward
model scores more highly. Since the reward model was trained to prefer
coherent, well-organised text over word-shuffled text, GRPO nudges the
policy toward more structured outputs. The improvement at nano scale is
real but modest. [The same pipeline on a 7B model with high-quality
human preference data produces the much larger step change]{.underline} in behaviour
seen in production-grade instruction-following models. The mechanism is
identical; the scale is different.

---

## Helper Functions

In [ ]:
def _download_tinyshakespeare(path: Path):
    import urllib.request
    path.parent.mkdir(parents=True, exist_ok=True)
    url = ('https://raw.githubusercontent.com/karpathy/char-rnn/'
           'master/data/tinyshakespeare/input.txt')
    print(f"  Downloading TinyShakespeare...")
    urllib.request.urlretrieve(url, path)
    print(f"  {path.stat().st_size / 1e6:.1f} MB downloaded.")


def _make_shakespeare_sft(raw: str, output: str, n_samples: int = 800):
    """Instruction dataset: synthetic prompts paired with Shakespeare excerpts."""
    random.seed(42)
    words   = raw.split()
    chunks  = [' '.join(words[i:i+80]) for i in range(0, len(words)-80, 80)]
    prompts = [
        "Write a short dramatic passage.",
        "Continue this scene in the style of Shakespeare.",
        "Write a soliloquy about fate.",
        "Write dialogue between two nobles.",
        "Write a verse about jealousy.",
        "Describe a battle in Shakespearean prose.",
        "Write a monologue about ambition.",
        "Compose a lament for a fallen king.",
    ]
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    with open(output, 'w') as f:
        for i in range(min(n_samples, len(chunks))):
            sample = {
                'messages': [
                    {'role': 'system',
                     'content': 'You are a creative writing assistant.'},
                    {'role': 'user',
                     'content': random.choice(prompts)},
                    {'role': 'assistant',
                     'content': chunks[i]},
                ]
            }
            f.write(json.dumps(sample) + '\n')
    print(f"  SFT dataset: {min(n_samples, len(chunks))} samples → {output}")


def _make_preference_data(raw: str, output: str, n_samples: int = 400):
    """
    Preference pairs: real Shakespeare excerpt (chosen) vs
    word-shuffled version of the same excerpt (rejected).
    """
    random.seed(42)
    words   = raw.split()
    chunks  = [' '.join(words[i:i+60]) for i in range(0, len(words)-60, 60)]
    prompts = [
        "Write a short passage in the style of Shakespeare.",
        "Write a dramatic scene.",
        "Compose a verse.",
    ]
    Path(output).parent.mkdir(parents=True, exist_ok=True)
    with open(output, 'w') as f:
        for i in range(min(n_samples, len(chunks))):
            chosen   = chunks[i]
            w        = chosen.split()
            random.shuffle(w)
            rejected = ' '.join(w)
            sample   = {
                'prompt':   random.choice(prompts),
                'chosen':   chosen,
                'rejected': rejected,
            }
            f.write(json.dumps(sample) + '\n')
    print(f"  Preference data: {min(n_samples, len(chunks))} pairs → {output}")


def _make_grpo_prompts() -> list[str]:
    return [
        "What should Hamlet do about his father's murder?",
        "Write a brief speech about the nature of ambition.",
        "Describe the relationship between power and corruption.",
        "What is the right way to treat a defeated enemy?",
        "Write about the consequences of unchecked jealousy.",
        "What makes a true leader in times of war?",
        "Write about the nature of loyalty and betrayal.",
        "What is the cost of revenge?",
        "Describe the burden of a guilty conscience.",
        "Write about the inevitability of fate.",
        "What separates a hero from a villain?",
        "Write a reflection on the passage of time.",
        "What is the price of ambition without wisdom?",
        "Describe what honour means to a soldier.",
        "Write about the loneliness of power.",
        "What does it mean to be truly brave?",
    ]

---

## The Main Entrypoint

In [ ]:
def main():
    import argparse

    parser = argparse.ArgumentParser(
        description='Nano reasoning model — end-to-end training pipeline.'
    )
    parser.add_argument(
        '--config', choices=['pico', 'nano'], default='pico',
        help=(
            'pico: sanity check on a laptop (~6 min CPU). '
            'nano: full run on a GPU (~20–50 min).'
        )
    )
    parser.add_argument('--run-dir',        default=None)
    parser.add_argument('--data-dir',       default='data')
    parser.add_argument('--pretrain-steps', type=int, default=None)
    parser.add_argument('--sft-steps',      type=int, default=None)
    parser.add_argument('--rm-steps',       type=int, default=None)
    parser.add_argument('--grpo-steps',     type=int, default=None)
    parser.add_argument('--no-quantize',    action='store_true')
    parser.add_argument('--no-speculative', action='store_true')
    parser.add_argument('--seed',           type=int, default=42)
    args = parser.parse_args()

    cfg = pico_config() if args.config == 'pico' else nano_config()

    # Apply any CLI overrides on top of the selected config
    if args.run_dir        is not None: cfg.run_dir        = args.run_dir
    if args.pretrain_steps is not None: cfg.pretrain_steps = args.pretrain_steps
    if args.sft_steps      is not None: cfg.sft_steps      = args.sft_steps
    if args.rm_steps       is not None: cfg.rm_steps       = args.rm_steps
    if args.grpo_steps     is not None: cfg.grpo_steps     = args.grpo_steps
    if args.no_quantize:    cfg.quantize        = False
    if args.no_speculative: cfg.use_speculative = False
    cfg.data_dir = args.data_dir
    cfg.seed     = args.seed

    torch.manual_seed(cfg.seed)
    random.seed(cfg.seed)
    np.random.seed(cfg.seed)

    Path(cfg.run_dir).mkdir(parents=True, exist_ok=True)
    Path(cfg.data_dir).mkdir(parents=True, exist_ok=True)
    cfg.save(f'{cfg.run_dir}/config.json')

    device = torch.device(cfg.device)
    print(f"\n{'═'*60}")
    print(f"  Nano Reasoning Model — {args.config.upper()} config")
    print(f"{'═'*60}")
    print(f"  Device  : {device}")
    print(f"  Run dir : {cfg.run_dir}")
    print(f"  Model   : d={cfg.model.d_model}  "
          f"L={cfg.model.n_layers}  "
          f"H={cfg.model.n_heads}  "
          f"V={cfg.model.vocab_size}")
    print(f"  Steps   : pretrain={cfg.pretrain_steps}  "
          f"sft={cfg.sft_steps}  "
          f"rm={cfg.rm_steps}  "
          f"grpo={cfg.grpo_steps}")
    print(f"{'═'*60}")

    # Tokenizer — trained once per run_dir and cached
    tok_path = f'{cfg.run_dir}/tokenizer.json'
    if Path(tok_path).exists():
        tok = Tokenizer.load(tok_path)
        print(f"\n  Tokenizer: loaded ({tok.vocab_size} tokens)")
    else:
        raw_path = Path(cfg.data_dir) / 'tinyshakespeare.txt'
        if not raw_path.exists():
            _download_tinyshakespeare(raw_path)
        raw = raw_path.read_text()
        tok = Tokenizer()
        tok.train(raw, vocab_size=cfg.model.vocab_size)
        tok.save(tok_path)
        print(f"\n  Tokenizer: trained ({tok.vocab_size} tokens) → {tok_path}")

    pretrain_ckpt = stage_pretrain(cfg, tok)
    sft_ckpt      = stage_sft(cfg, tok, pretrain_ckpt)
    rm_ckpt       = stage_reward_model(cfg, tok, sft_ckpt)
    grpo_ckpt     = stage_grpo(cfg, tok, sft_ckpt, rm_ckpt)
    engine        = stage_deploy(cfg, tok, grpo_ckpt)

    stage_evaluate(cfg, tok, engine, pretrain_ckpt, sft_ckpt, grpo_ckpt)


if __name__ == '__main__':
    main()

---

## Running the Pipeline

```bash
# Step 1 — verify everything works on your laptop
python train_nano_reasoning.py --config pico

# Step 2 — full run on a GPU
python train_nano_reasoning.py --config nano

# Stages that have existing checkpoints are skipped automatically,
# so you can re-run after a crash and pick up where you left off
python train_nano_reasoning.py --config nano

# Override step counts without changing the config
python train_nano_reasoning.py --config nano --pretrain-steps 10000

# Disable speculative decoding (useful on CPU or MPS)
python train_nano_reasoning.py --config nano --no-speculative
```

Approximate runtimes:

| Config | Hardware | Pretrain | SFT | RM | GRPO | Total |
|---|---|---|---|---|---|---|
| pico | Laptop CPU | ~3 min | ~1 min | <1 min | <1 min | ~6 min |
| nano | A100 40GB | ~8 min | ~3 min | ~2 min | ~5 min | ~20 min |
| nano | RTX 3090 | ~25 min | ~8 min | ~5 min | ~12 min | ~50 min |
| nano | M2 MacBook | ~90 min | ~25 min | ~15 min | ~35 min | ~2.7 hr |

The recommended workflow: run `pico` first on your development machine,
confirm the six pico checks in the section above all pass, then run `nano`
on your GPU. [Because each stage saves a checkpoint and skips if one exists,
progress is preserved if a run is interrupted.]{.underline}

---

## Extensions

The following directions are natural continuations from here.

**Scaling.** The architecture, training loop, and alignment pipeline scale
directly. Increasing parameter count and data volume — while keeping the
same structural decisions — reproduces the behaviour of production-scale
models. The engineering constraints change; the principles do not.

**Data.** TinyShakespeare is a convenient corpus for demonstration. Real
pretraining corpora consist of filtered web text, books, code, and
scientific literature at the terabyte scale. The data pipeline from
Tutorial 7 — MinHash deduplication, uint16 shards, reservoir shuffling
— was designed with that scale in mind.

**Alignment.** DPO and GRPO represent current standard practice. Active
research directions include constitutional AI, process reward models,
debate-based oversight, and scalable supervision. Each extends the
reward-model-plus-policy-optimisation framework from Tutorials 11–13.

**Inference.** Tutorial 16 covered the primary single-GPU optimisations.
Production serving introduces continuous batching, paged attention (vLLM),
and tensor parallelism across multiple devices.

**Multimodality.** Vision-language models use the same Transformer backbone.
The additional components are a vision encoder (typically a ViT) and a
projection layer connecting visual tokens to the language model's input
space. The training and alignment procedures are otherwise identical.

---

## Final Exercises

**1.** Run `pico` first and confirm all six checks pass. Then run `nano`.
Produce the three-way qualitative comparison and write a paragraph
explaining, in terms of the training dynamics from Tutorials 11–13,
why the GRPO model's responses differ from the SFT model's.

**2.** Swap DPO (Tutorial 12) in for GRPO in Stage 4. Run both versions of
`nano` and compare final reward scores on held-out prompts and perplexity
on held-out Shakespeare. Under what conditions does GRPO outperform DPO
on this dataset? Consider sample efficiency and response diversity.

**3.** Implement a pipeline profiler: measure wall-clock time and peak GPU
memory for each stage. Identify the bottleneck. Then apply one
optimisation — gradient checkpointing for pretraining, LoRA rank
reduction for SFT, batch size tuning for the reward model, or group
size reduction for GRPO — and re-benchmark.

**4.** Extend the GRPO prompts with simple logical reasoning questions
("If A is taller than B and B is taller than C, who is shortest?").
Run `nano`. Measure whether the reward model's coherence signal is
sufficient to improve logical reasoning, or whether a task-specific
reward (exact-match on the answer) is required. This is the central
question of reward design in RLHF.

**5.** For each adjacent stage pair (pretrained→SFT, SFT→GRPO), compute
the per-layer Frobenius norm of the weight change:
$\|W_{\text{after}} - W_{\text{before}}\|_F / \|W_{\text{before}}\|_F$.
Plot this as a heatmap over layers. Which layers change most during SFT?
Which during GRPO? Consider what this implies about which layers encode
syntax, semantics, and task behaviour respectively.

**6.** Run `nano` three times with different random seeds. Report variance
in final reward, perplexity, and throughput. Identify which stage is
most sensitive to initialisation. Then apply one variance-reduction
technique — checkpoint averaging, reward model ensembling, or GRPO
learning rate reduction — and measure its effect.

---

## Autoresearch Exercises

These exercises ask you to treat the pretraining stage as a search
problem: systematically propose changes, measure their effect on a
fixed metric, and keep only the improvements. This is the same loop
that produced the chart at the top of this section — 83 experiments,
15 kept, validation BPB falling from 0.998 to 0.977 over the course
of the search.

The reference script is `train.py`. It is a self-contained, time-budgeted
pretraining run on a single GPU. The metric is validation bits-per-byte[^bpb]
(BPB, lower is better). One experiment = one full run to completion within
the time budget. You propose one change per experiment, measure BPB, and
keep the change only if BPB improves.

[**Why one change at a time?**]{.underline} Because two simultaneous changes that both
individually improve BPB can interact — one may cancel or amplify the
other. Holding everything else fixed is the only way to know what caused
an improvement. This is the discipline that makes autoresearch reproducible
rather than lucky.

**On keeping records.** Before you start, add three lines to the bottom
of `train.py` that write results to a JSONL log:

[^bpb]: Bits-per-byte (BPB) normalises next-token log-loss by the number of UTF-8 bytes rather than tokens, making it tokeniser-independent. BPB = $\log_2(e) \times \text{NLL}$. For reference, a uniform 256-symbol model scores 8.0 BPB; GPT-2 scores ≈ 1.4 BPB on the WebText test set; very strong models reach ≈ 0.9 BPB on diverse English text.

In [ ]:
import json, pathlib
result = {"experiment": EXPERIMENT_NAME, "val_bpb": val_bpb,
          "kept": None,  # fill in manually after comparison
          "notes": ""}
pathlib.Path("results.jsonl").open("a").write(json.dumps(result) + "\n")

Fill in `kept` (True/False) and `notes` after each run. After ten
experiments, a table will reveal more than any single result. The
progress chart in this tutorial was produced from exactly this kind
of log.

---

**Exercise A — Establish your baseline.**

Run `train.py` without any changes. Record `val_bpb`, `training_seconds`,
`peak_vram_mb`, and `mfu_percent` from the final summary block. Label
this `experiment_000_baseline` in your log. Every subsequent experiment
is measured against this number.

*Pointer.* If you do not have a stable baseline you cannot interpret
anything. Do not skip this step, even if it feels like wasted GPU time.
Baseline variance across seeds is also worth measuring: run the same
config twice with `torch.manual_seed(137)` and compare. If BPB differs
by more than 0.001, your time budget is too short for stable comparisons
and you should increase `TIME_BUDGET` before proceeding.

---

**Exercise B — Map the sensitivity of the learning rate schedule.**

The script uses three schedule parameters: `WARMUP_RATIO`, `WARMDOWN_RATIO`,
and `FINAL_LR_FRAC`. Run three experiments, changing one parameter at a
time:

1. `WARMDOWN_RATIO`: try `0.3`, `0.5` (baseline), `0.7`
2. `WARMUP_RATIO`: try `0.0` (baseline), `0.05`, `0.10`
3. `FINAL_LR_FRAC`: try `0.0` (baseline), `0.05`, `0.1`

Plot BPB vs parameter value for each sweep. Which parameter has the
largest effect? Which direction is it? Keep the best setting before
moving to the next exercise.

*Pointer.* Cooldown ratio typically matters more than warmup at this
scale. If you see BPB improving as you extend cooldown, you are probably
compute-limited — the model is still learning at the end of the run and
cutting the LR earlier lets it consolidate. This is the same observation
as the "warmdown 0.5→0.7" improvement visible in the progress chart.

---

**Exercise C — Search over depth and aspect ratio.**

The model size is determined by two numbers: `DEPTH` (number of layers)
and `ASPECT_RATIO` (model_dim = depth × aspect_ratio). The current
baseline uses `DEPTH=8, ASPECT_RATIO=64`, giving a model_dim of 512.

Within a fixed FLOPs budget (same `TIME_BUDGET`), deeper-narrower vs
shallower-wider models perform differently. Run the following grid,
keeping total parameter count approximately constant by adjusting
`ASPECT_RATIO` as you change `DEPTH`:

| DEPTH | ASPECT_RATIO | approx model_dim |
|-------|-------------|-----------------|
| 6 | 86 | 512 |
| 8 | 64 | 512 (baseline) |
| 9 | 57 | 512 |
| 12 | 43 | 512 |

Note that `build_model_config` rounds `model_dim` up to the nearest
multiple of `HEAD_DIM`, so actual sizes will vary slightly. Record exact
parameter counts from the printed summary.

*Pointer.* Depth tends to matter more than width for language modelling
at fixed compute, up to a point. Very deep models can underfit within a
short time budget because each forward pass is slower. The "depth 9
aspect_ratio 57" improvement in the progress chart suggests the optimal
depth for this budget is slightly above 8 — but the return diminishes
past that, as the discarded experiments around depth 12 show.

---

**Exercise D — Tune the sliding window pattern.**

The `WINDOW_PATTERN` string controls which layers use short sliding window
attention (`S`) and which use full context (`L`). The current baseline
is `"SSSL"`, which tiles as SSSL-SSSL across all 8 layers with the last
layer always forced to full context.

Try the following patterns and measure BPB for each:

```
"SSSL"   # baseline — 3 short : 1 long
"SSL"    # 2 short : 1 long (more full-context layers)
"SSSL"   # baseline
"SSSSL"  # 4 short : 1 long (fewer full-context layers)
"L"      # all full context — useful as a reference point
```

Also try moving the long-context layers to different positions within the
pattern, e.g. `"LSSS"` vs `"SSSL"` vs `"SLSS"`.

*Pointer.* The window pattern interacts with sequence length. At shorter
sequences the distinction between `S` and `L` is smaller (half of a
short sequence is still a short sequence). At longer sequences, full-context
layers become expensive and the window pattern is doing more work. If your
results are noisy, check whether the effective short window size
(`sequence_len // 2`) is actually meaningfully smaller than the sequence
length for your dataset.

---

**Exercise E — Value embedding ablation.**

The model uses value embeddings (ResFormer-style): on alternating layers,
the value vectors are augmented with a learned embedding of the input
token, gated by a small linear layer. This adds parameters and compute.

Ablate it by setting `has_ve` to always return `False`:

```python
def has_ve(layer_idx, n_layer):
    return False   # ablation: no value embeddings
```

Compare BPB with and without. Then try a partial ablation: value embeddings
on every layer instead of alternating layers. Record the parameter count
change alongside BPB — value embeddings add `n_ve_layers × vocab_size × kv_dim`
parameters, which can be substantial.

*Pointer.* Value embeddings help most when the model is undertrained
relative to its parameter count (i.e. the embedding table is underutilised).
At short time budgets you may see little effect; at longer budgets the
gap is more likely to appear. If you have time, run the ablation at two
different `TIME_BUDGET` values and compare the gap.

---

**Exercise F — Implement the autoresearch loop.**

The exercises above have been manual: propose, run, record, decide.
Now automate the decision loop. Write a Python script `autoresearch.py`
that:

1. Maintains a `current_best` config (a dict of hyperparameter values).
2. Proposes a candidate config by perturbing one hyperparameter at a time.
   Start with a fixed list of searchable parameters and perturbation ranges.
3. Writes the candidate config into a temporary copy of `train.py`, runs
   it as a subprocess, and parses `val_bpb` from stdout.
4. Compares the candidate BPB to `current_best`. If it improves by more
   than a threshold (e.g. 0.0005), update `current_best` and log the
   improvement as "kept". Otherwise log it as "discarded".
5. Repeats until a budget (number of experiments or wall time) is exhausted.
6. At the end, plots the running best BPB over experiment number, with
   kept improvements labelled, reproducing the style of the progress chart
   above.

The searchable parameters and suggested ranges to start with:

In [ ]:
SEARCH_SPACE = {
    'DEPTH':           [6, 7, 8, 9, 10, 12],
    'ASPECT_RATIO':    [48, 56, 64, 72, 80],
    'WARMDOWN_RATIO':  [0.3, 0.4, 0.5, 0.6, 0.7],
    'WARMUP_RATIO':    [0.0, 0.03, 0.05, 0.10],
    'EMBEDDING_LR':    [0.3, 0.6, 0.8, 1.0],
    'UNEMBEDDING_LR':  [0.002, 0.004, 0.006, 0.008],
    'WINDOW_PATTERN':  ["SSL", "SSSL", "SSSL", "SSSSL"],
    'TOTAL_BATCH_SIZE': [2**18, 2**19, 2**20],
}

For the perturbation strategy, use **one-at-a-time greedy search**:
pick the parameter whose best candidate (across all values in its range)
gives the largest BPB reduction. Keep that value, then move to the next
parameter. Cycle until no parameter can improve BPB by more than the
threshold.

*Pointer on subprocess management.* Use `subprocess.run` with
`capture_output=True` to run each experiment and parse its output.
Add a timeout equal to `TIME_BUDGET × 1.5` to catch hangs. Kill and
discard any run that exceeds the timeout or prints "FAIL". A simple
approach to config injection is to write a `config.py` that
`train.py` imports, and have `autoresearch.py` overwrite `config.py`
before each run.

*Pointer on the acceptance threshold.* The progress chart shows that
early improvements (steps 0–10) are large and late improvements
(steps 60–83) are small. This is normal: easy wins come first. As
the search matures, lower the acceptance threshold from 0.001 to
0.0005 to 0.0002 to keep finding improvements. Alternatively, use
a fixed threshold but report the improvement as a fraction of the
remaining gap to some target BPB.

*Pointer on reproducibility.* Each experiment should use the same
random seed unless you are explicitly searching over seeds. The seed
is a hyperparameter like any other: `torch.manual_seed(42)` locks in
one trajectory; `torch.manual_seed(137)` gives you another. The
"random seed 42→137" improvement at the end of the progress chart is
a real phenomenon — some seeds produce better loss landscapes for a
given config — but it should be the last thing you tune, not the first.

---

**Exercise G — Extend the search space with architectural changes.**

Once your autoresearch loop is working, add the following architectural
variants to the search space. Each requires a small code change in
addition to a config change.

**Squared ReLU vs GELU.** The baseline MLP uses `F.relu(x).square()`.
Try replacing it with `F.gelu(x)`:

```python
# In MLP.forward:
x = F.gelu(x)          # candidate
# x = F.relu(x).square()  # baseline
```

Add a boolean `USE_SQUARED_RELU` flag and make it searchable.

**GQA (Grouped Query Attention).** The baseline uses full multi-head
attention (`n_kv_head = n_head`). GQA reduces KV cache size by sharing
key/value heads across groups of query heads. Try `n_kv_head = n_head // 2`
and `n_kv_head = n_head // 4`. The architecture already supports this
via the `n_kv_head` field in `GPTConfig` — you just need to expose it
in the search space.

**RoPE base frequency.** The baseline uses `base=10000` in
`_precompute_rotary_embeddings`. Larger base values extend effective
context length. Try `base` values of `10000`, `50000`, `100000`,
`200000`. This is the same search visible in the progress chart
(the "RoPE base frequency" experiments near the end), where the optimum
turned out to be around 100000–200000 for the sequence length used.

For each variant, measure: (1) BPB change vs baseline, (2) change in
tokens/sec (some variants are slower), and (3) change in peak VRAM.
An improvement in BPB that costs 10% in throughput is not obviously
a win — you need to hold compute constant to compare fairly.

---

*End of series.*